In [1]:
from rapidsegment import StrategicSegmentBuilder, StrategicSegmentScore, UniversalDataLoader


In [2]:
data = UniversalDataLoader(file_path = r"/workspaces/RapidSegment/student_burnout_dropout_dataset_2.csv").load()

In [3]:
import pyarrow as pa

# Dictionary comprehension to get counts for all columns
null_counts = {name: data[name].null_count for name in data.column_names}
print(null_counts)
# Output example: {'col1': 0, 'col2': 3, 'col3': 1}


{'Student_ID': 0, 'Age': 0, 'Gender': 0, 'Year_of_Study': 0, 'Department': 0, 'Residence_Type': 0, 'Attendance_Percent': 51, 'Study_Hours_Per_Day': 35, 'Previous_GPA': 35, 'Backlogs': 15, 'Sleep_Hours': 35, 'Screen_Time_Hours': 36, 'Exercise_Freq_Per_Week': 0, 'Social_Activity_Score': 40, 'Part_Time_Job': 0, 'Family_Income_Bracket': 0, 'Financial_Stress_Score': 43, 'Family_Support_Score': 44, 'Stress_Level': 49, 'Anxiety_Score': 39, 'Motivation_Score': 41, 'Peer_Pressure_Score': 41, 'Counseling_Access': 0, 'Burnout_Level': 0, 'Dropout_Risk': 0}


In [4]:

import pyarrow.compute as pc


# 2. Define your replacements
condition = pc.equal(data["Dropout_Risk"], "Yes")
new_values = pc.if_else (condition, 1, 0)

# 3. Replace the old column with the new computed array in the immutable table
new_table = data.set_column(
    data.schema.get_field_index("Dropout_Risk"), 
    "Dropout_Risk", 
    new_values
)


In [5]:
grid_config = {
    "min_sample_size": [10, 50],
    "min_lift": [2.0, 1.2]
}


In [6]:
builder = StrategicSegmentBuilder(
    target="Dropout_Risk",
    min_sample_size=50,
    min_events=5,
    top_n_vars=10,
    max_segments=10,
    max_feature_reuse=1,
    param_grid=grid_config,
    enable_diversity=True,
    feature_groups=None,
    ignore_features=["Student_ID"]
)

In [7]:
segments_summary = builder.extract_segments(new_table)

2026-07-16 14:51:55,788 | INFO     | [builder.py:354] | DuckDB Configured: Threads=4/4, MemoryLimit=7GB
2026-07-16 14:51:55,807 | INFO     | [builder.py:369] | Dynamic Grid Search Enabled: 4 total configurations.
2026-07-16 14:51:55,808 | INFO     | [builder.py:390] | Iteration 1 | Remaining Volume: 800 | Base Rate: 54.62%
2026-07-16 14:51:57,041 | INFO     | [builder.py:539] | Feature Usage Tracker Update -> 'Study_Hours_Per_Day' used count = 1
2026-07-16 14:51:57,042 | INFO     | [builder.py:539] | Feature Usage Tracker Update -> 'Family_Income_Bracket' used count = 1
2026-07-16 14:51:57,042 | INFO     | [builder.py:539] | Feature Usage Tracker Update -> 'Burnout_Level' used count = 1
2026-07-16 14:51:57,043 | INFO     | [builder.py:554] | Segment 1 Captured (Size Floor: 50 | Lift Floor: 1.2): (Study_Hours_Per_Day >= 0.05 AND Study_Hours_Per_Day < 3.45) AND Family_Income_Bracket IN ('Lower-Middle') AND Burnout_Level IN ('High')
2026-07-16 14:51:57,053 | INFO     | [builder.py:390] | 

In [8]:
import pandas as pd
segments_df = pd.DataFrame(segments_summary)
print("\nExtracted Segment Profiles:")
segments_df[["segment_id", "count", "lift", "meta_applied_sample_size", "sql_filter"]]



Extracted Segment Profiles:


,segment_id,count,lift,meta_applied_sample_size,sql_filter
0,1,51,1.758873,50,(Study_Hours_Per_Day >= 0.05 AND Study_Hours_P...
1,2,90,1.333161,50,Attendance_Percent < 52.45
2,3,93,1.286440,50,Stress_Level >= 7.45


In [9]:
coverage_report = builder.evaluate_final_coverage(new_table)
print("\nCascading Portfolio Coverage Analysis:")
pd.DataFrame(coverage_report)



Cascading Portfolio Coverage Analysis:


,segment,total_count,target_events,response_rate,base_response_rate,capture_rate,lift
0,0,611,291.0,47.626841,54.625,76.375,0.871887
1,1,51,49.0,96.078431,54.625,6.375,1.758873
2,2,39,33.0,84.615385,54.625,4.875,1.549023
3,3,99,64.0,64.646465,54.625,12.375,1.183459


In [10]:
import duckdb
con = duckdb.connect(database='temp_scoring.db', read_only=False)

In [11]:
con.execute("CREATE TABLE IF NOT EXISTS scored_data AS SELECT Student_ID, Dropout_Risk FROM new_table")

In [12]:
print("--- FULL SEGMENT RULES ---\n")

for index, row in pd.DataFrame(segments_df).iterrows():
    print(f"Segment ID: {row['segment_id']}")
    print(f"Raw Rule:   {row['rule_string']}")
    print(f"SQL Filter: {row['sql_filter']}")
    print("-" * 50)

--- FULL SEGMENT RULES ---

Segment ID: 1
Raw Rule:   Study_Hours_Per_Day=[0.05, 3.45) & Family_Income_Bracket=<ArrowStringArray>
['Lower-Middle']
Length: 1, dtype: str & Burnout_Level=<ArrowStringArray>
['High']
Length: 1, dtype: str
SQL Filter: (Study_Hours_Per_Day >= 0.05 AND Study_Hours_Per_Day < 3.45) AND Family_Income_Bracket IN ('Lower-Middle') AND Burnout_Level IN ('High')
--------------------------------------------------
Segment ID: 2
Raw Rule:   Attendance_Percent=(-inf, 52.45)
SQL Filter: Attendance_Percent < 52.45
--------------------------------------------------
Segment ID: 3
Raw Rule:   Stress_Level=[7.45, inf)
SQL Filter: Stress_Level >= 7.45
--------------------------------------------------


In [13]:
con.close()